# Brazilian E-Commerce Analysis

## Olist Ecommerce Dataset

This notebook extends the PostgreSQL and SQL analysis with Python-based data validation, exploratory data analysis, distribution analysis, outlier investigation, visualization, and business interpretation.

**Reporting period:** January 1, 2017 through August 31, 2018  
**Primary KPI scope:** Successfully delivered orders  
**Revenue definition:** Product item value excluding freight unless otherwise stated

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Locate the project root whether the notebook starts from
# the repository root or from the notebooks folder.
current_directory = Path.cwd().resolve()

if (current_directory / "README.md").exists():
    PROJECT_ROOT = current_directory
elif (current_directory.parent / "README.md").exists():
    PROJECT_ROOT = current_directory.parent
else:
    raise FileNotFoundError(
        "Project root could not be identified. Open the repository folder in VS Code."
    )


RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
SQL_EXPORT_DIR = PROJECT_ROOT / "outputs" / "sql_exports"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sql_export_files = sorted(SQL_EXPORT_DIR.glob("*.csv"))


print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data folder exists: {RAW_DATA_DIR.exists()}")
print(f"SQL export folder exists: {SQL_EXPORT_DIR.exists()}")
print(f"Figure folder exists: {FIGURE_DIR.exists()}")
print(f"SQL CSV files found: {len(sql_export_files)}")

Python version: 3.14.6
Pandas version: 3.0.3
Project root: /Users/ivana/Projects/ecommerce-data-analysis
Raw data folder exists: True
SQL export folder exists: True
Figure folder exists: True
SQL CSV files found: 12


## 1. Data Loading and Initial Validation

The analysis begins by loading the four core datasets required to evaluate orders, revenue, customer purchasing behavior, delivery performance, reviews, and customer geography.

The source files are validated before loading to ensure that the notebook fails clearly if any required dataset is missing.

In [10]:
required_raw_files = {
    "orders": RAW_DATA_DIR / "olist_orders_dataset.csv",
    "order_items": RAW_DATA_DIR / "olist_order_items_dataset.csv",
    "customers": RAW_DATA_DIR / "olist_customers_dataset.csv",
    "reviews": RAW_DATA_DIR / "olist_order_reviews_dataset.csv",
}

missing_files = [
    path.name
    for path in required_raw_files.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required raw files are missing: "
        + ", ".join(missing_files)
    )

print("All required raw datasets were found:\n")

for dataset_name, file_path in required_raw_files.items():
    print(f"{dataset_name:<12} {file_path.name}")

All required raw datasets were found:

orders       olist_orders_dataset.csv
order_items  olist_order_items_dataset.csv
customers    olist_customers_dataset.csv
reviews      olist_order_reviews_dataset.csv


In [11]:
orders = pd.read_csv(
    required_raw_files["orders"],
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

order_items = pd.read_csv(
    required_raw_files["order_items"],
    parse_dates=["shipping_limit_date"],
)

customers = pd.read_csv(
    required_raw_files["customers"],
)

reviews = pd.read_csv(
    required_raw_files["reviews"],
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp",
    ],
)

core_datasets = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "reviews": reviews,
}

data_inventory = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "rows": dataframe.shape[0],
            "columns": dataframe.shape[1],
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "memory_mb": round(
                dataframe.memory_usage(deep=True).sum() / 1_048_576,
                2,
            ),
        }
        for dataset_name, dataframe in core_datasets.items()
    ]
)

data_inventory

,dataset,rows,columns,duplicate_rows,memory_mb
0,orders,99441,8,0,24.66
1,order_items,112650,7,0,29.54
2,customers,99441,5,0,26.59
3,reviews,99224,7,0,27.76


### 1.1 Column Quality and Data Grain

Before creating analytical datasets, the source tables are profiled for data types, missing values, unique values, candidate-key duplication, and relationship integrity.

Missing values are not automatically treated as errors. For example, incomplete or cancelled orders may not have delivery timestamps, while review comments are optional.

In [12]:
def build_column_profile(dataset_name, dataframe):
    """Create a column-level data quality summary."""
    return pd.DataFrame(
        {
            "dataset": dataset_name,
            "column": dataframe.columns,
            "dtype": dataframe.dtypes.astype(str).to_numpy(),
            "missing_values": dataframe.isna().sum().to_numpy(),
            "missing_pct": (
                dataframe.isna()
                .mean()
                .mul(100)
                .round(2)
                .to_numpy()
            ),
            "unique_values": dataframe.nunique(dropna=True).to_numpy(),
        }
    )


column_profile = pd.concat(
    [
        build_column_profile(dataset_name, dataframe)
        for dataset_name, dataframe in core_datasets.items()
    ],
    ignore_index=True,
)

column_profile

,dataset,column,dtype,missing_values,missing_pct,unique_values
0,orders,order_id,str,0,0.00,99441
1,orders,customer_id,str,0,0.00,99441
2,orders,order_status,str,0,0.00,8
3,orders,order_purchase_timestamp,datetime64[us],0,0.00,98875
4,orders,order_approved_at,datetime64[us],160,0.16,90733
5,orders,order_delivered_carrier_date,datetime64[us],1783,1.79,81018
6,orders,order_delivered_customer_date,datetime64[us],2965,2.98,95664
7,orders,order_estimated_delivery_date,datetime64[us],0,0.00,459
8,order_items,order_id,str,0,0.00,98666
9,order_items,order_item_id,int64,0,0.00,21


In [13]:
def check_candidate_key(dataset_name, dataframe, key_columns):
    """Evaluate whether one or more columns uniquely identify each row."""
    duplicate_key_rows = int(
        dataframe.duplicated(
            subset=key_columns,
            keep=False,
        ).sum()
    )

    null_key_rows = int(
        dataframe[key_columns]
        .isna()
        .any(axis=1)
        .sum()
    )

    distinct_keys = int(
        dataframe[key_columns]
        .drop_duplicates()
        .shape[0]
    )

    return {
        "dataset": dataset_name,
        "candidate_key": " + ".join(key_columns),
        "rows": len(dataframe),
        "distinct_keys": distinct_keys,
        "duplicate_key_rows": duplicate_key_rows,
        "null_key_rows": null_key_rows,
        "is_unique_key": (
            duplicate_key_rows == 0
            and null_key_rows == 0
        ),
    }


grain_checks = pd.DataFrame(
    [
        check_candidate_key(
            "orders",
            orders,
            ["order_id"],
        ),
        check_candidate_key(
            "order_items",
            order_items,
            ["order_id", "order_item_id"],
        ),
        check_candidate_key(
            "customers",
            customers,
            ["customer_id"],
        ),
        check_candidate_key(
            "customers",
            customers,
            ["customer_unique_id"],
        ),
        check_candidate_key(
            "reviews",
            reviews,
            ["review_id"],
        ),
        check_candidate_key(
            "reviews",
            reviews,
            ["order_id"],
        ),
    ]
)

grain_checks

,dataset,candidate_key,rows,distinct_keys,duplicate_key_rows,null_key_rows,is_unique_key
0,orders,order_id,99441,99441,0,0,True
1,order_items,order_id + order_item_id,112650,112650,0,0,True
2,customers,customer_id,99441,99441,0,0,True
3,customers,customer_unique_id,99441,96096,6342,0,False
4,reviews,review_id,99224,98410,1603,0,False
5,reviews,order_id,99224,98673,1098,0,False


In [14]:
def check_relationship(
    relationship_name,
    child_dataframe,
    child_column,
    parent_dataframe,
    parent_column,
):
    """Check whether child-table identifiers exist in the parent table."""
    unmatched_mask = ~child_dataframe[child_column].isin(
        parent_dataframe[parent_column]
    )

    return {
        "relationship": relationship_name,
        "child_rows": len(child_dataframe),
        "unmatched_rows": int(unmatched_mask.sum()),
        "unmatched_unique_keys": int(
            child_dataframe.loc[
                unmatched_mask,
                child_column,
            ].nunique()
        ),
    }


relationship_checks = pd.DataFrame(
    [
        check_relationship(
            "orders.customer_id → customers.customer_id",
            orders,
            "customer_id",
            customers,
            "customer_id",
        ),
        check_relationship(
            "order_items.order_id → orders.order_id",
            order_items,
            "order_id",
            orders,
            "order_id",
        ),
        check_relationship(
            "reviews.order_id → orders.order_id",
            reviews,
            "order_id",
            orders,
            "order_id",
        ),
    ]
)

relationship_checks

,relationship,child_rows,unmatched_rows,unmatched_unique_keys
0,orders.customer_id → customers.customer_id,99441,0,0
1,order_items.order_id → orders.order_id,112650,0,0
2,reviews.order_id → orders.order_id,99224,0,0


### 1.2 Review Table Grain Investigation

The reviews dataset is not strictly one row per review ID or one row per order. Before joining reviews to orders, duplicate identifiers and orders with multiple review records are investigated.

The final analytical dataset will use one review record per order, with multiple review scores aggregated consistently with the SQL analytical view.


In [15]:
review_rows_per_order = (
    reviews.groupby("order_id")
    .size()
)

review_grain_summary = pd.DataFrame(
    {
        "metric": [
            "Total review rows",
            "Unique review IDs",
            "Unique reviewed orders",
            "Rows with duplicated review IDs",
            "Rows belonging to duplicated order IDs",
            "Orders with multiple review rows",
            "Maximum review rows for one order",
        ],
        "value": [
            len(reviews),
            reviews["review_id"].nunique(),
            reviews["order_id"].nunique(),
            int(
                reviews.duplicated(
                    subset=["review_id"],
                    keep=False,
                ).sum()
            ),
            int(
                reviews.duplicated(
                    subset=["order_id"],
                    keep=False,
                ).sum()
            ),
            int((review_rows_per_order > 1).sum()),
            int(review_rows_per_order.max()),
        ],
    }
)

review_grain_summary

,metric,value
0,Total review rows,99224
1,Unique review IDs,98410
2,Unique reviewed orders,98673
3,Rows with duplicated review IDs,1603
4,Rows belonging to duplicated order IDs,1098
5,Orders with multiple review rows,547
6,Maximum review rows for one order,3


In [16]:
duplicate_review_id_analysis = (
    reviews.groupby("review_id")
    .agg(
        row_count=("review_id", "size"),
        unique_orders=("order_id", "nunique"),
        unique_review_scores=("review_score", "nunique"),
    )
    .query("row_count > 1")
    .reset_index()
)

duplicate_review_id_summary = pd.DataFrame(
    {
        "metric": [
            "Duplicated review IDs",
            "Duplicated review IDs linked to multiple orders",
            "Duplicated review IDs with different scores",
        ],
        "value": [
            len(duplicate_review_id_analysis),
            int(
                (
                    duplicate_review_id_analysis["unique_orders"] > 1
                ).sum()
            ),
            int(
                (
                    duplicate_review_id_analysis[
                        "unique_review_scores"
                    ] > 1
                ).sum()
            ),
        ],
    }
)

duplicate_review_id_summary

,metric,value
0,Duplicated review IDs,789
1,Duplicated review IDs linked to multiple orders,789
2,Duplicated review IDs with different scores,0


In [17]:
review_rows_per_order_distribution = (
    review_rows_per_order
    .value_counts()
    .sort_index()
    .rename_axis("review_rows_per_order")
    .reset_index(name="number_of_orders")
)

review_rows_per_order_distribution["order_share_pct"] = (
    review_rows_per_order_distribution["number_of_orders"]
    .div(review_rows_per_order_distribution["number_of_orders"].sum())
    .mul(100)
    .round(2)
)

review_rows_per_order_distribution

,review_rows_per_order,number_of_orders,order_share_pct
0,1,98126,99.45
1,2,543,0.55
2,3,4,0.00


### 1.3 Create an Order-Level Reviews Table

Because some orders contain multiple review records, the raw reviews table cannot be joined directly to the orders table without duplicating orders.

Reviews are therefore aggregated to one row per order. The average review score is used to remain consistent with the PostgreSQL analytical view and the completed SQL analysis. Review counts and score ranges are retained for validation and outlier investigation.

In [18]:
order_reviews = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        review_row_count=("review_id", "size"),
        unique_review_ids=("review_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        min_review_score=("review_score", "min"),
        max_review_score=("review_score", "max"),
        latest_review_answer_timestamp=(
            "review_answer_timestamp",
            "max",
        ),
    )
)

order_reviews["has_multiple_review_rows"] = (
    order_reviews["review_row_count"] > 1
)

order_reviews["has_conflicting_review_scores"] = (
    order_reviews["min_review_score"]
    != order_reviews["max_review_score"]
)

order_reviews.head()

,order_id,review_row_count,unique_review_ids,avg_review_score,min_review_score,max_review_score,latest_review_answer_timestamp,has_multiple_review_rows,has_conflicting_review_scores
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,5.0,5,5,2017-09-22 10:57:03,False,False
1,00018f77f2f0320c557190d7a144bdd3,1,1,4.0,4,4,2017-05-15 11:34:13,False,False
2,000229ec398224ef6ca0657da4fc703e,1,1,5.0,5,5,2018-01-23 16:06:31,False,False
3,00024acbcdf0a6daa1e931b038114c75,1,1,4.0,4,4,2018-08-15 16:39:01,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,5.0,5,5,2017-03-03 10:54:59,False,False


In [19]:
order_reviews_validation = pd.DataFrame(
    {
        "metric": [
            "Rows in order-level reviews table",
            "Unique order IDs",
            "Duplicate order IDs",
            "Orders with multiple review rows",
            "Orders with conflicting review scores",
            "Missing average review scores",
            "Minimum average review score",
            "Maximum average review score",
        ],
        "value": [
            len(order_reviews),
            order_reviews["order_id"].nunique(),
            int(
                order_reviews.duplicated(
                    subset=["order_id"]
                ).sum()
            ),
            int(
                order_reviews[
                    "has_multiple_review_rows"
                ].sum()
            ),
            int(
                order_reviews[
                    "has_conflicting_review_scores"
                ].sum()
            ),
            int(
                order_reviews[
                    "avg_review_score"
                ].isna()
                .sum()
            ),
            order_reviews["avg_review_score"].min(),
            order_reviews["avg_review_score"].max(),
        ],
    }
)

order_reviews_validation

,metric,value
0,Rows in order-level reviews table,98673.0
1,Unique order IDs,98673.0
2,Duplicate order IDs,0.0
3,Orders with multiple review rows,547.0
4,Orders with conflicting review scores,202.0
5,Missing average review scores,0.0
6,Minimum average review score,1.0
7,Maximum average review score,5.0


In [20]:
multiple_review_orders = (
    order_reviews.loc[
        order_reviews["has_multiple_review_rows"],
        [
            "order_id",
            "review_row_count",
            "unique_review_ids",
            "avg_review_score",
            "min_review_score",
            "max_review_score",
            "has_conflicting_review_scores",
        ],
    ]
    .sort_values(
        by=[
            "review_row_count",
            "has_conflicting_review_scores",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

multiple_review_orders.head(10)

,order_id,review_row_count,unique_review_ids,avg_review_score,min_review_score,max_review_score,has_conflicting_review_scores
0,03c939fd7fd3b38f8485a0f95798f1f6,3,3,3.333333,3,4,True
1,c88b1d1b157a9999ce368f218a407141,3,3,4.333333,3,5,True
2,8e17072ec97ce29f0e1f111e598b0c85,3,3,1.000000,1,1,False
3,df56136b8031ecd28e200bb18e6ddb2e,3,3,5.000000,5,5,False
4,013056cfe49763c6f66bda03396c5ee3,2,2,4.500000,4,5,True
5,02355020fd0a40a0d56df9f6ff060413,2,2,2.000000,1,3,True
6,029863af4b968de1e5d6a82782e662f5,2,2,4.500000,4,5,True
7,03eba6d9fef8f5b3e811d4b5a7cca9cd,2,2,4.500000,4,5,True
8,04f1827088d972a62224f5203a071500,2,2,3.000000,1,5,True
9,0544030711e50ec2cb6c15764d22891a,2,2,2.500000,1,4,True


### 1.4 Create an Order-Level Sales Table

The order-items dataset contains one row per product item, so orders containing multiple products appear more than once.

To prevent duplicated orders and overstated revenue, order items are aggregated to one row per order before being joined to the orders table. Product revenue and freight are calculated separately, and the total order value includes both components. 

Although the PostgreSQL reporting layer includes `vw_order_details`, that view retains order-item-level grain because each product item is represented separately. For Python distribution and outlier analysis, the raw order-items dataset is independently aggregated to one row per order. This also provides a separate validation that product revenue and freight totals reconcile exactly with the source data.

In [25]:
order_sales = (
    order_items.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        product_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
        minimum_item_price=("price", "min"),
        maximum_item_price=("price", "max"),
    )
)

order_sales["total_order_value"] = (
    order_sales["product_revenue"]
    + order_sales["freight_value"]
)

order_sales.head()

,order_id,item_count,unique_products,unique_sellers,product_revenue,freight_value,minimum_item_price,maximum_item_price,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29,58.90,58.90,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93,239.90,239.90,259.83
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87,199.00,199.00,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,12.99,12.99,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14,199.90,199.90,218.04


In [22]:
order_sales_validation = pd.DataFrame(
    {
        "metric": [
            "Order-item source rows",
            "Rows in order-level sales table",
            "Unique order IDs",
            "Duplicate order IDs",
            "Orders with multiple items",
            "Maximum items in one order",
            "Source product revenue",
            "Aggregated product revenue",
            "Revenue difference",
            "Source freight total",
            "Aggregated freight total",
            "Freight difference",
        ],
        "value": [
            len(order_items),
            len(order_sales),
            order_sales["order_id"].nunique(),
            int(
                order_sales.duplicated(
                    subset=["order_id"]
                ).sum()
            ),
            int((order_sales["item_count"] > 1).sum()),
            int(order_sales["item_count"].max()),
            order_items["price"].sum(),
            order_sales["product_revenue"].sum(),
            (
                order_sales["product_revenue"].sum()
                - order_items["price"].sum()
            ),
            order_items["freight_value"].sum(),
            order_sales["freight_value"].sum(),
            (
                order_sales["freight_value"].sum()
                - order_items["freight_value"].sum()
            ),
        ],
    }
)

order_sales_validation

,metric,value
0,Order-item source rows,112650.00
1,Rows in order-level sales table,98666.00
2,Unique order IDs,98666.00
3,Duplicate order IDs,0.00
4,Orders with multiple items,9803.00
5,Maximum items in one order,21.00
6,Source product revenue,13591643.70
7,Aggregated product revenue,13591643.70
8,Revenue difference,0.00
9,Source freight total,2251909.54


In [23]:
sales_reconciliation = pd.DataFrame(
    {
        "measure": [
            "product_revenue",
            "freight_value",
        ],
        "source_total": [
            order_items["price"].sum(),
            order_items["freight_value"].sum(),
        ],
        "aggregated_total": [
            order_sales["product_revenue"].sum(),
            order_sales["freight_value"].sum(),
        ],
    }
)

sales_reconciliation["difference"] = (
    sales_reconciliation["aggregated_total"]
    - sales_reconciliation["source_total"]
)

sales_reconciliation["matches_source"] = np.isclose(
    sales_reconciliation["source_total"],
    sales_reconciliation["aggregated_total"],
)

sales_reconciliation

,measure,source_total,aggregated_total,difference,matches_source
0,product_revenue,13591643.70,13591643.70,0.0,True
1,freight_value,2251909.54,2251909.54,0.0,True


In [24]:
largest_orders_by_item_count = (
    order_sales[
        [
            "order_id",
            "item_count",
            "unique_products",
            "unique_sellers",
            "product_revenue",
            "freight_value",
            "total_order_value",
        ]
    ]
    .sort_values(
        by=[
            "item_count",
            "total_order_value",
        ],
        ascending=[False, False],
    )
    .head(10)
    .reset_index(drop=True)
)

largest_orders_by_item_count

,order_id,item_count,unique_products,unique_sellers,product_revenue,freight_value,total_order_value
0,8272b63d03f5f79c56e9e4120aec44ef,21,3,1,31.80,164.37,196.17
1,ab14fdcfbe524636d65ee38360e22ce8,20,1,1,1974.00,288.80,2262.80
2,1b15974a0141d54e36626dca3fdc731a,20,1,1,2000.00,202.40,2202.40
3,428a2f660dc84138d969ccd69a0ab6d5,15,1,1,982.35,243.30,1225.65
4,9ef13efd6949e4573a18964dd1bbe7f5,15,1,1,765.00,18.00,783.00
5,73c8ab38f07dc94389065f7eba4f297a,14,1,1,826.00,188.02,1014.02
6,9bdc4d4c71aa1de4606060929dee888c,14,1,1,419.86,108.92,528.78
7,37ee401157a3a0b28c9c6d0ed8c3b24b,13,1,1,389.87,96.07,485.94
8,3a213fcdfe7d98be74ea0dc05a8b31ae,12,1,1,1296.00,186.24,1482.24
9,637617b3ffe9e2f7a2411243829226d0,12,4,1,958.80,288.15,1246.95


## 2. Build the Order-Level Analytical Dataset

The validated source tables are combined into one analytical dataset containing one row per order.

Customer information is joined using `customer_id`, while the aggregated sales and review tables are joined using `order_id`. One-to-one merge validation is applied to prevent accidental duplication.

Orders without matching item records or reviews are retained so that missing analytical coverage remains visible. The primary KPI dataset is then limited to successfully delivered orders with item-level sales data during the reporting period.

In [26]:
customer_attributes = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
    ]
].copy()


orders_analytical = (
    orders.merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        order_sales,
        on="order_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        order_reviews,
        on="order_id",
        how="left",
        validate="one_to_one",
    )
)


orders_analytical["has_order_items"] = (
    orders_analytical["item_count"].notna()
)

orders_analytical["has_review"] = (
    orders_analytical["avg_review_score"].notna()
)

orders_analytical["is_delivered"] = (
    orders_analytical["order_status"].eq("delivered")
)

orders_analytical["in_reporting_period"] = (
    orders_analytical["order_purchase_timestamp"]
    .ge(pd.Timestamp("2017-01-01"))
    &
    orders_analytical["order_purchase_timestamp"]
    .lt(pd.Timestamp("2018-09-01"))
)

orders_analytical["included_in_primary_kpis"] = (
    orders_analytical["is_delivered"]
    & orders_analytical["has_order_items"]
    & orders_analytical["in_reporting_period"]
)


orders_analytical["delivery_days"] = (
    orders_analytical[
        "order_delivered_customer_date"
    ].dt.normalize()
    -
    orders_analytical[
        "order_purchase_timestamp"
    ].dt.normalize()
).dt.days.astype("Int64")


orders_analytical["delivery_vs_estimate_days"] = (
    orders_analytical[
        "order_delivered_customer_date"
    ].dt.normalize()
    -
    orders_analytical[
        "order_estimated_delivery_date"
    ].dt.normalize()
).dt.days.astype("Int64")


orders_analytical.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,latest_review_answer_timestamp,has_multiple_review_rows,has_conflicting_review_scores,has_order_items,has_review,is_delivered,in_reporting_period,included_in_primary_kpis,delivery_days,delivery_vs_estimate_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,2017-10-12 03:43:48,False,False,True,True,True,True,True,8,-8
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,2018-08-08 18:37:50,False,False,True,True,True,True,True,14,-6
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,2018-08-22 19:07:58,False,False,True,True,True,True,True,9,-18
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,2017-12-05 19:21:58,False,False,True,True,True,True,True,14,-13
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,2018-02-18 13:02:51,False,False,True,True,True,True,True,3,-10


In [27]:
analytical_dataset_validation = pd.DataFrame(
    {
        "metric": [
            "Rows in source orders table",
            "Rows in analytical dataset",
            "Unique order IDs",
            "Duplicate order IDs",
            "Orders missing customer information",
            "Orders with item records",
            "Orders without item records",
            "Orders with review data",
            "Orders without review data",
            "Orders included in primary KPIs",
        ],
        "value": [
            len(orders),
            len(orders_analytical),
            orders_analytical["order_id"].nunique(),
            int(
                orders_analytical.duplicated(
                    subset=["order_id"]
                ).sum()
            ),
            int(
                orders_analytical[
                    "customer_unique_id"
                ].isna().sum()
            ),
            int(
                orders_analytical[
                    "has_order_items"
                ].sum()
            ),
            int(
                (
                    ~orders_analytical[
                        "has_order_items"
                    ]
                ).sum()
            ),
            int(
                orders_analytical[
                    "has_review"
                ].sum()
            ),
            int(
                (
                    ~orders_analytical[
                        "has_review"
                    ]
                ).sum()
            ),
            int(
                orders_analytical[
                    "included_in_primary_kpis"
                ].sum()
            ),
        ],
    }
)

analytical_dataset_validation

,metric,value
0,Rows in source orders table,99441
1,Rows in analytical dataset,99441
2,Unique order IDs,99441
3,Duplicate order IDs,0
4,Orders missing customer information,0
5,Orders with item records,98666
6,Orders without item records,775
7,Orders with review data,98673
8,Orders without review data,768
9,Orders included in primary KPIs,96211


In [28]:
delivered_orders_analysis = (
    orders_analytical.loc[
        orders_analytical[
            "included_in_primary_kpis"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Rows in delivered-orders analysis dataset:",
    f"{len(delivered_orders_analysis):,}",
)

print(
    "Unique orders:",
    f"{delivered_orders_analysis['order_id'].nunique():,}",
)

print(
    "Unique customers:",
    f"{delivered_orders_analysis['customer_unique_id'].nunique():,}",
)

Rows in delivered-orders analysis dataset: 96,211
Unique orders: 96,211
Unique customers: 93,104


In [29]:
sql_sales_summary = pd.read_csv(
    SQL_EXPORT_DIR
    / "delivered_sales_summary.csv"
)

python_sales_summary = {
    "delivered_orders": len(
        delivered_orders_analysis
    ),
    "delivered_revenue": round(
        delivered_orders_analysis[
            "product_revenue"
        ].sum(),
        2,
    ),
    "delivered_freight": round(
        delivered_orders_analysis[
            "freight_value"
        ].sum(),
        2,
    ),
    "delivered_total_value_including_freight": round(
        delivered_orders_analysis[
            "total_order_value"
        ].sum(),
        2,
    ),
    "delivered_average_order_value": round(
        delivered_orders_analysis[
            "product_revenue"
        ].mean(),
        2,
    ),
}


sql_sales_values = (
    sql_sales_summary.iloc[0]
    .to_dict()
)


sales_sql_reconciliation = pd.DataFrame(
    {
        "metric": list(
            python_sales_summary.keys()
        ),
        "python_value": list(
            python_sales_summary.values()
        ),
        "sql_value": [
            sql_sales_values[metric]
            for metric in python_sales_summary
        ],
    }
)


sales_sql_reconciliation["difference"] = (
    sales_sql_reconciliation["python_value"]
    - sales_sql_reconciliation["sql_value"]
)


sales_sql_reconciliation["matches_sql"] = np.isclose(
    sales_sql_reconciliation["python_value"],
    sales_sql_reconciliation["sql_value"],
    atol=0.01,
)


sales_sql_reconciliation

,metric,python_value,sql_value,difference,matches_sql
0,delivered_orders,96211.00,96211.00,0.0,True
1,delivered_revenue,13181027.13,13181027.13,0.0,True
2,delivered_freight,2192092.88,2192092.88,0.0,True
3,delivered_total_value_including_freight,15373120.01,15373120.01,0.0,True
4,delivered_average_order_value,137.00,137.00,0.0,True


### Validation Result

The merged Python dataset preserves one row per order and contains no duplicated order IDs. The independently calculated delivered-order count, product revenue, freight, total value, and average order value match the final PostgreSQL results.

This confirms that the Python transformation preserves the analytical grain and does not duplicate revenue.